Text Processing

In [ ]:
import csv
import re

def clean_file(input_path, output_path):
    # load
    with open(input_path, 'r', encoding='utf-8') as infile:
        lines = infile.readlines()

    valid_rows = []

    # check
    for line in lines:
        columns = [col.strip() for col in re.split(r',\s*(?!\s)', line)]

        # check the row columns
        if len(columns) == 11:
            valid_rows.append(columns)

    # write
    with open(output_path, 'w', newline='', encoding='utf-8') as outfile:
        writer = csv.writer(outfile)
        writer.writerows(valid_rows)

    return f"File cleaned and saved to {output_path}"

# location
input_file = 'XX.txt'
output_file = 'XX.csv'

# process
clean_file(input_file, output_file)

Clean

In [ ]:
import pandas as pd
DATA01 = pd.read_csv('XX.csv')

DATA02 = DATA01[DATA01.iloc[:, 2].str.contains(r'\d', na=False)] # PRESS
DATA03 = DATA02[DATA02.iloc[:, 4].str.contains(r'\d', na=False)] # PUBLICATION YEAR
DATA04 = DATA03[DATA03.iloc[:, 5].str.contains(r'\d', na=False)] # TEMPERATURE
DATA05 = DATA04[DATA04.iloc[:, 6].str.contains(r'\d', na=False)] # REACTION TIME
DATA06 = DATA05[DATA05.iloc[:, 8].str.contains(r'\d', na=False)] # YIELD
DATA07 = DATA06[DATA06.iloc[:, 9].str.contains(r'\d', na=False)] # SELECTIVITY
DATA08 = DATA07[DATA07.iloc[:, 10].str.contains(r'\d', na=False)] # CONVERSION

In [ ]:
def extract_and_average(df, column_name):
    
    

    def process_value(value):
        
        if pd.isna(value):
            return None  
        value = re.sub(r'[<>%]', '', str(value))
        match = re.match(r'(\d+)-(\d+)', str(value))  
        if match:
            m, n = int(match.group(1)), int(match.group(2))
            return (m + n) / 2  
        else:
            try:
                
                return float(value)
            except ValueError:
                return None  

    
    df[column_name] = df[column_name].apply(process_value)
    
    
    return df

def remove_rows_greater_than_100(data, column_name):


    
    df_cleaned =data[data[column_name] <= 100]

    return df_cleaned

def remove_duplicate_rows_based_on_columns(DATA, columns_to_check, comparison_column, comparison_column2, comparison_column3):

    
    df_cleaned01 = DATA.loc[DATA.groupby(columns_to_check)[comparison_column].idxmax()]
    df_cleaned02 = df_cleaned01.loc[df_cleaned01.groupby(columns_to_check)[comparison_column2].idxmax()]
    df_cleaned03 = df_cleaned01.loc[df_cleaned01.groupby(columns_to_check)[comparison_column3].idxmax()]


    return df_cleaned01, df_cleaned02, df_cleaned03

def remove_rows_with_nan_in_columns(DATA, columns_to_check):

    
    df_cleaned = DATA.dropna(subset=columns_to_check)

    return df_cleaned


In [ ]:

NUM01 = extract_and_average(DATA08, 'target_product_yield')
NUM02 = extract_and_average(NUM01, 'target_product_selectivity')
NUM03 = extract_and_average(NUM02, 'target_product_conversion')
NUM04 = extract_and_average(NUM03, 'reaction_temperature')
NUM05 = extract_and_average(NUM04, 'reaction_time')
NUM06 = extract_and_average(NUM05, 'reaction_pressure')
CHECK01 = remove_rows_greater_than_100(NUM06, 'target_product_conversion')
CHECK02 = remove_rows_greater_than_100(CHECK01, 'target_product_selectivity')
columns_to_check = ['catalyst_material', 'oxidant', 'target_product', 'liquid_composition_and_conditions', 'reaction_pressure', 'reaction_temperature', 'reaction_time'] 
comparison_column = 'publication_year'  
comparison_column2 = 'target_product_selectivity'
comparison_column3 = 'target_product_conversion'
YEAR01, YEAR02, YEAR03 = remove_duplicate_rows_based_on_columns(CHECK02, columns_to_check, comparison_column, comparison_column2, comparison_column3)
columns_to_check = ['catalyst_material', 'oxidant', 'target_product', 'reaction_pressure', 'reaction_temperature', 'reaction_time']  
NAS_C = remove_rows_with_nan_in_columns(YEAR03, columns_to_check)
#NAS_C.to_csv('NAS_C.csv', index=False)